In [1]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

In [ ]:
INPUT_CSV = "./cm_coords.csv"
OUTPUT_CSV = "./output/coordenadas_pvout.csv"
BATCH_SIZE = 10

In [9]:
async def get_pvout(page, lat, lon):
    url = f"https://globalsolaratlas.info/map?c={lat},{lon},6&s={lat},{lon}&m=site"
    await page.goto(url, wait_until="networkidle", timeout=60000)
    try:
        await page.wait_for_selector("gsa-site-data-item", timeout=20000)
        items = await page.query_selector_all("gsa-site-data-item")
        for item in items:
            key_el = await item.query_selector("gsa-site-data-key")
            if key_el and "PVOUT" in await key_el.inner_text():
                value_el = await item.query_selector("sg-unit-value-inner")
                if value_el:
                    return float((await value_el.inner_text()).strip().split("\n")[0])
    except Exception as e:
        print(f"  Error ({lat}, {lon}): {e}")
    return None

async def main():
    df = pd.read_csv(INPUT_CSV).head(BATCH_SIZE).copy()
    df["PVOUT_kWh_kWp"] = None

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for i, row in df.iterrows():
            lat, lon = row["Latitud"], row["Longitud"]
            print(f"[{i+1}/{len(df)}] ({lat}, {lon})")
            pvout = await get_pvout(page, lat, lon)
            df.at[i, "PVOUT_kWh_kWp"] = pvout
            print(f"  {'✓ ' + str(pvout) + ' kWh/kWp' if pvout else '✗ Sin resultado'}")
            await asyncio.sleep(2)

        await browser.close()

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nGuardado en {OUTPUT_CSV}")
    print(df[["Código Modular", "Latitud", "Longitud", "PVOUT_kWh_kWp"]])

import nest_asyncio
nest_asyncio.apply()
asyncio.run(main())

[1/10] (-16.440842, -71.577918)
  ✓ 2027.4 kWh/kWp
[2/10] (-16.42022, -71.54042)
  ✓ 2032.6 kWh/kWp
[3/10] (-16.4116, -71.5248)
  ✓ 2030.7 kWh/kWp
[4/10] (-16.4116, -71.5248)
  ✓ 2030.7 kWh/kWp
[5/10] (-16.4116, -71.5248)
  ✓ 2030.7 kWh/kWp
[6/10] (-16.4102, -71.52335)


/home/rodrigo/Documents/ml_playground/.venv/lib64/python3.14/site-packages/pyee/base.py:194: RuntimeWarning: coroutine 'main' was never awaited
  funcs = list(self._events.get(event, OrderedDict()).values())


  ✓ 2030.7 kWh/kWp
[7/10] (-16.39186, -71.53059)
  ✓ 2031.6 kWh/kWp
[8/10] (-16.40718, -71.53606)
  ✓ 2030.6 kWh/kWp
[9/10] (-16.40718, -71.53606)
  ✓ 2030.6 kWh/kWp
[10/10] (-16.40876, -71.52288)
  ✓ 2030.7 kWh/kWp

Guardado en ../data/coordenadas_pvout.csv
   Código Modular    Latitud   Longitud PVOUT_kWh_kWp
0         3994635 -16.440842 -71.577918        2027.4
1         1752989 -16.420220 -71.540420        2032.6
2         1237908 -16.411600 -71.524800        2030.7
3         1030410 -16.411600 -71.524800        2030.7
4         1030402 -16.411600 -71.524800        2030.7
5         1721521 -16.410200 -71.523350        2030.7
6          226423 -16.391860 -71.530590        2031.6
7          226431 -16.407180 -71.536060        2030.6
8         1397454 -16.407180 -71.536060        2030.6
9         1399286 -16.408760 -71.522880        2030.7
